# 00 — PDF Extraction

Converts PDFs into:
1. **Full markdown** (`md_docs/*.md`) — for chunking & embedding
2. **Reference metadata JSON** (`md_ref/*.json`) — title, authors, year, DOI, etc.

Uses:
- **Docling** for OCR + layout-aware markdown export
- **GPT-4.1-mini** to extract structured reference metadata from the first 5 pages

> Run this notebook **once per batch of new PDFs**.  
> Already-processed files (with existing `.json`) are skipped automatically.

## 1 — Imports & Config

In [ ]:
import time
import json
from os import environ
from pathlib import Path
from dotenv import load_dotenv
from loguru import logger
from openai import OpenAI

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableStructureOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# --- Monkey-patch: allow URIs without scheme (e.g. "www.example.com/...") ---
# Docling's PdfHyperlink uses Pydantic AnyUrl which rejects bare hostnames.
# This causes entire page batches to fail during preprocessing.
# Fix: relax uri field to Optional[str].  (Tracked in docling-core PR #520)
from typing import Optional
from docling_core.types.doc.page import PdfHyperlink, SegmentedPdfPage
PdfHyperlink.__annotations__["uri"] = Optional[str]
PdfHyperlink.model_fields["uri"].annotation = Optional[str]
PdfHyperlink.model_rebuild(force=True)
SegmentedPdfPage.model_rebuild(force=True)
# ---------------------------------------------------------------------------

load_dotenv()

API_KEY = environ["OPENAI_API_KEY"]

import sys
sys.path.insert(0, str(Path("__file__").resolve().parent.parent / "src"))
from scientific_rag.config import RAW_DIR, MD_DIR, REF_DIR, EXTRACT_PROMPT_PATH

PDF_DIR         = RAW_DIR
PROMPT_PATH     = EXTRACT_PROMPT_PATH
MD_OUTPUT_DIR   = MD_DIR
JSON_OUTPUT_DIR = REF_DIR

MODEL  = "gpt-4.1-mini"
client = OpenAI(api_key=API_KEY, timeout=1500)

print(f"PDF source     : {PDF_DIR}")
print(f"Markdown output: {MD_OUTPUT_DIR}")
print(f"JSON output    : {JSON_OUTPUT_DIR}")

## 2 — Helper Functions

In [ ]:
def read_txt(path: Path) -> str:
    return path.read_text(encoding="utf-8")


def ensure_dirs():
    MD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("Output directories ready.")

## 3 — Docling OCR Converter

In [ ]:
def build_doc_converter():
    """
    Configure Docling PDF converter with OCR and table extraction.
    """
    options = PdfPipelineOptions()
    options.do_ocr = True
    options.do_table_structure = True
    options.table_structure_options = TableStructureOptions(do_cell_matching=True)
    options.ocr_options.lang = ["en"]
    options.accelerator_options = AcceleratorOptions(
        num_threads=4,
        device=AcceleratorDevice.AUTO
    )

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=options)
        }
    )


def extract_pdf(pdf_path: Path, converter: DocumentConverter):
    """Run OCR on a single PDF and return the conversion result."""
    start = time.time()
    result = converter.convert(pdf_path)
    logger.info(f"OCR finished in {time.time() - start:.2f}s")
    return result


def get_pages_1_to_5_markdown(conv_result) -> str:
    """Export only the first 5 pages as markdown (for reference extraction)."""
    pages = []
    for page_no in sorted(conv_result.document.pages):
        if page_no > 5:
            break
        pages.append(conv_result.document.export_to_markdown(page_no=page_no))
    return "\n\n".join(pages)


print("OCR functions defined.")

## 4 — LLM Reference Extraction

In [ ]:
def get_reference_json(ocr_text: str, prompt_template: str) -> dict:
    """
    Send the first 5 pages to the LLM and extract structured reference metadata.
    """
    prompt = prompt_template.replace("{OCR_TEXT}", ocr_text)

    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=0,
        text={"format": {"type": "json_object"}},
    )

    return json.loads(response.output_text)


print("LLM extraction function defined.")

## 5 — Single-PDF Processing Pipeline

In [ ]:
def process_pdf(pdf_path: Path, converter, prompt_template):
    """
    Full pipeline for one PDF:
      1. Skip if JSON already exists
      2. OCR → full markdown
      3. Extract first 5 pages → LLM → reference JSON
    """
    stem = pdf_path.stem
    json_path = JSON_OUTPUT_DIR / f"{stem}.json"
    md_path   = MD_OUTPUT_DIR   / f"{stem}.md"

    if json_path.exists():
        logger.info(f"Skipping (already processed): {stem}")
        return

    logger.info(f"Processing: {pdf_path.name}")

    # OCR full document
    conv = extract_pdf(pdf_path, converter)

    # Save full markdown
    md_path.write_text(conv.document.export_to_markdown(), encoding="utf-8")

    # Extract reference metadata from first 5 pages
    ocr_text = get_pages_1_to_5_markdown(conv)
    reference = get_reference_json(ocr_text, prompt_template)

    # Save reference JSON
    json_path.write_text(
        json.dumps(reference, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    logger.info(f"Completed: {stem}")


print("process_pdf() defined.")

## 6 — Batch Process All PDFs

In [ ]:
ensure_dirs()

prompt_template = read_txt(PROMPT_PATH)
converter = build_doc_converter()

pdfs = sorted(PDF_DIR.glob("*.pdf"))
logger.info(f"Found {len(pdfs)} PDFs in {PDF_DIR}")

for pdf in pdfs:
    try:
        process_pdf(pdf, converter, prompt_template)
    except Exception:
        logger.exception(f"Failed on {pdf.name}")

logger.info("All documents processed.")

## 7 — Verify Outputs

In [ ]:
md_files   = sorted(MD_OUTPUT_DIR.glob("*.md"))
json_files = sorted(JSON_OUTPUT_DIR.glob("*.json"))

print(f"Markdown files : {len(md_files)}")
print(f"JSON files     : {len(json_files)}")

# Show first few
print("\n── Sample markdown files ──")
for f in md_files[:5]:
    print(f"  {f.name}")

print("\n── Sample JSON files ──")
for f in json_files[:5]:
    print(f"  {f.name}")

## 8 — Inspect a Reference JSON

Pick any stem to preview its extracted metadata.

In [ ]:
import json

# Change this to any document stem you want to inspect
STEM = "001. The Joint Winter Runway Friction NASA Perspective"

json_path = JSON_OUTPUT_DIR / f"{STEM}.json"

if json_path.exists():
    meta = json.loads(json_path.read_text(encoding="utf-8"))
    print(f"Title   : {meta.get('title')}")
    print(f"Year    : {meta.get('year')}")
    print(f"Authors : {meta.get('authors')}")
    print(f"DOI     : {meta.get('doi')}")
    print(f"Type    : {meta.get('type')}")
else:
    print(f"File not found: {json_path}")

---

✅ **Done.** Next step: open `01_indexing.ipynb` to chunk, embed, and index into Qdrant.